In [147]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder,OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [148]:
df = pd.read_csv("..\insurance.csv")

In [149]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [150]:
x = df.drop('charges',axis=1)
y=df['charges']

In [151]:
cat_col = x.select_dtypes(include='object').columns
num_col = x.select_dtypes(include='number').columns
num_col

Index(['age', 'bmi', 'children'], dtype='object')

In [152]:
cat_col
ord_col = ['smoker']
one_col = ['sex','region']

In [153]:
ord_pipeline = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=[['no','yes']]))
])

one_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder',OneHotEncoder(handle_unknown='ignore'))
])

num_pipeline = Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='median')),
    ('sclaer', StandardScaler())
])

preprossor = ColumnTransformer(transformers=[
    ('numeric', num_pipeline, num_col),
    ('ordinal',ord_pipeline, ord_col),
    ('onehot', one_pipeline, one_col)
])

In [154]:
x_train, x_test, y_train, y_test = train_test_split(x,y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    )

In [155]:
x_train = preprossor.fit_transform(x_train)
x_test = preprossor.transform(x_test)

In [156]:
# if hasattr(x_train, 'toarray'):
#     x_train = x_train.toarray()

# if hasattr(x_test, 'toarray'):
#     x_test = x_test.toarray()

# x_train = np.asarray(x_train, dtype=np.float32)
# x_test = np.asarray(x_test, dtype=np.float32)

In [157]:
y_scaler = StandardScaler()

y_train = y_scaler.fit_transform(
    y_train.to_numpy().reshape(-1, 1)
)

y_test = y_scaler.transform(
y_test.to_numpy().reshape(-1, 1)
)

In [158]:
model = keras.Sequential([
    layers.Input(shape=(x_train.shape[1],)),

    layers.Dense(64),
    layers.LeakyReLU(negative_slope=0.1),

    layers.Dense(32),
    layers.LeakyReLU(negative_slope=0.1),

    layers.Dense(1, activation='linear')
])

In [159]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mean_squared_error',
    metrics=['mae']
)

model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_14 (Dense)                │ (None, 64)             │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_4 (LeakyReLU)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_5 (LeakyReLU)       │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,817 (11.00 KB)

 Trainable params: 2,817 (11.00 KB)

 Non-trainable params: 0 (0.00 B)

In [160]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

In [161]:
history = model.fit(
    x_train,
    y_train,
    epochs=300,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/300


27/27 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - loss: 0.9243 - mae: 0.7456 - val_loss: 0.7671 - val_mae: 0.7031
Epoch 2/300
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.6075 - mae: 0.6100 - val_loss: 0.5096 - val_mae: 0.5210
Epoch 3/300
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.3813 - mae: 0.4549 - val_loss: 0.3111 - val_mae: 0.3871
Epoch 4/300
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.2510 - mae: 0.3605 - val_loss: 0.2240 - val_mae: 0.3015
Epoch 5/300
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.2075 - mae: 0.3100 - val_loss: 0.1951 - val_mae: 0.2963
Epoch 6/300
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.1872 - mae: 0.2905 - val_loss: 0.1775 - val_mae: 0.2777
Epoch 7/300
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.1740 - mae: 0.2663 - val_loss: 0.1665 - val_mae: 0.2667
Epoch 8/300
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.1680 - mae: 0.2551 - val_loss: 0.1587 - val_mae: 0.2457
Epoch 9/300
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.1652 - mae: 0.

In [162]:
loss, mae = model.evaluate(x_test, y_test)

print("Test Loss:", loss)
print("Test MAE:", mae)

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1513 - mae: 0.2610 
Test Loss: 0.1513265073299408
Test MAE: 0.2609943449497223


In [163]:
y_pred = model.predict(x_test)


9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step


In [164]:
print("R2 Score:", r2_score(y_test, y_pred))

R2 Score: 0.8593128249991734
